# Chapter 4 - Training Video Generation Models

Companion notebook for *Video Generation with AI* (O'Reilly) by Joseph Enochs.

**Learning objective:** Build the complete training pipeline for a Latte-style video diffusion model, from configuration management and video data loading through the training loop, loss functions, and distributed scaling.


In [ ]:
# Install dependencies not preinstalled on Colab.
# torch, numpy, opencv, and matplotlib already ship with the Colab runtime.
%pip install -q omegaconf h5py

## Demo configuration

**Scale-up cell.** The settings below keep every training demo tiny so the whole notebook runs top to bottom in a few minutes, even on CPU. Real video model training uses the values in the comments and needs serious GPU time (see the chapter's hardware notes).


In [ ]:
# Tiny demo settings so the notebook runs end to end quickly.
# The comments show realistic values for actual training runs.
DEMO = {
    "num_videos": 4,       # synthetic clips to generate (real: thousands to millions)
    "video_frames": 60,    # frames per synthetic source video
    "frame_size": 64,      # spatial resolution      (real: 256 or higher)
    "clip_length": 8,      # frames per training clip (real: 16)
    "frame_skip": 2,       # temporal stride          (real: 4)
    "batch_size": 2,       # per-step batch           (real: 4+ with accumulation)
    "grad_accum": 2,       # gradient accumulation    (real: 8)
    "train_steps": 12,     # optimizer steps to run   (real: 1_000_000)
    "warmup_steps": 4,     # LR warmup                (real: 10_000)
    "learning_rate": 2e-4, # demo LR                  (real: 1e-4)
}

In [ ]:
import json
import os
from pathlib import Path

import cv2
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {device}")

WORK_DIR = Path("./chapter4_workspace")
WORK_DIR.mkdir(exist_ok=True)

## 1. Configuration-Driven Training

Every experiment must be fully specified, versioned, and reproducible. We use hierarchical YAML files as experiment blueprints and OmegaConf to load and manage them, mirroring Latte's approach.

### YAML configuration structure

The configuration separates concerns: model architecture, training hyperparameters, dataset settings, diffusion schedule, and logging.


In [ ]:
# config/train_video.yaml - Main training configuration
# This mirrors the configuration structure used in Latte-style production systems
CONFIG_YAML = '''
model:
  name: "Latte-XL/2"
  image_size: 256
  num_frames: 16
  frame_interval: 3
  latent_size: 32
  num_layers: 24
  num_heads: 16
  hidden_size: 1152
  patch_size: 2
  # VAE configuration
  vae_model: "stabilityai/sd-vae-ft-ema"
  # Text encoder
  text_encoder: "t5-base"

training:
  batch_size: 4
  gradient_accumulation_steps: 8
  learning_rate: 1e-4
  warmup_steps: 10000
  max_steps: 1000000
  # Optimization
  optimizer: "adamw"
  weight_decay: 0.01
  beta1: 0.9
  beta2: 0.999
  epsilon: 1e-8
  # Gradient management
  gradient_clip: 1.0
  mixed_precision: true
  tf32: true  # Latte's TF32 optimization
  # EMA
  use_ema: true
  ema_decay: 0.9999

dataset:
  name: "ucf101"
  data_path: "/path/to/ucf101"
  cache_dir: "/path/to/cache"
  num_workers: 8
  # Video sampling
  sample_rate: 4
  clip_duration: 2.0
  # Augmentation
  random_flip: true
  color_jitter: 0.1

diffusion:
  timesteps: 1000
  beta_schedule: "linear"
  beta_start: 0.0001
  beta_end: 0.02
  # Loss configuration
  loss_type: "mse"
  use_min_snr: false  # Start simple like Latte

logging:
  output_dir: "./outputs"
  logging_steps: 100
  save_steps: 5000
  validation_steps: 1000
  tensorboard: true
  wandb:
    project: "video-diffusion"
    entity: "my-team"
'''

config_dir = WORK_DIR / "configs"
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "train_video.yaml"
config_path.write_text(CONFIG_YAML)
print(f"Wrote {config_path}")

### Loading and managing configurations

OmegaConf gives us automatic type conversion, variable interpolation, command-line overrides, and immutable configs.


In [ ]:
# Configuration management with OmegaConf
from omegaconf import OmegaConf, DictConfig


def load_config(config_path: str, overrides: list = None) -> DictConfig:
    """
    Load configuration with OmegaConf, supporting overrides.
    This mirrors the pattern used for Latte-style training.
    """
    # Load base configuration
    base_config = OmegaConf.load(config_path)

    # Apply command-line overrides if provided
    if overrides:
        override_config = OmegaConf.from_dotlist(overrides)
        config = OmegaConf.merge(base_config, override_config)
    else:
        config = base_config

    # Resolve interpolations (${model.hidden_size} references)
    OmegaConf.resolve(config)

    # Make configuration immutable to prevent accidental changes
    OmegaConf.set_struct(config, True)

    return config


# Load the config, then load it again with an override, exactly as a
# command line flag like "--override training.batch_size=8" would do.
config = load_config(config_path)
print(f"Model: {config.model.name}, LR: {config.training.learning_rate}")

config_override = load_config(config_path, overrides=["training.batch_size=8"])
print(f"Overridden batch size: {config_override.training.batch_size}")

In [ ]:
def setup_training_from_config(config: DictConfig):
    """
    Initialize the training environment from configuration.
    """
    # Enable TF32 for better performance (Latte optimization)
    if config.training.tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    # Create output directory
    output_dir = Path(config.logging.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save configuration for reproducibility
    OmegaConf.save(config, output_dir / "config.yaml")

    return output_dir


output_dir = setup_training_from_config(config)
print(f"Run directory ready: {output_dir}, config saved for reproducibility")

## 2. Dataset Preparation and Loading

Video datasets are massive and heterogeneous, and frames can only be decoded sequentially. The pipeline below solves this with clip indexing, HDF5 caching, and temporally consistent augmentation.

### Synthetic demo videos

Real datasets like WebVid or UCF-101 are far too large for a notebook, so we generate a few tiny videos of moving squares plus a metadata JSON in the exact format the dataset class expects.


In [ ]:
def make_synthetic_videos(video_dir, num_videos, num_frames, size):
    """Write small videos of moving colored squares plus metadata.json."""
    video_dir = Path(video_dir)
    video_dir.mkdir(exist_ok=True, parents=True)

    colors = [(255, 60, 60), (60, 255, 60), (60, 60, 255), (255, 255, 60),
              (255, 60, 255), (60, 255, 255)]
    names = ["red", "green", "blue", "yellow", "magenta", "cyan"]

    metadata = []
    for i in range(num_videos):
        path = video_dir / f"video_{i}.avi"
        # MJPG in an AVI container decodes reliably everywhere
        writer = cv2.VideoWriter(
            str(path), cv2.VideoWriter_fourcc(*"MJPG"), 16, (size, size)
        )
        color = colors[i % len(colors)]
        for f in range(num_frames):
            frame = np.zeros((size, size, 3), dtype=np.uint8)
            # Square slides across the frame over time
            x = int((f / num_frames) * (size - 16))
            y = size // 2 - 8 + int(8 * np.sin(f / 6))
            frame[y:y + 16, x:x + 16] = color[::-1]  # BGR for OpenCV
            writer.write(frame)
        writer.release()

        metadata.append({
            "path": path.name,
            "num_frames": num_frames,
            "caption": f"a {names[i % len(names)]} square moving across the screen",
            "id": f"synthetic_{i}",
        })

    with open(video_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    return video_dir / "metadata.json"


video_dir = WORK_DIR / "videos"
metadata_path = make_synthetic_videos(
    video_dir, DEMO["num_videos"], DEMO["video_frames"], DEMO["frame_size"]
)
print(f"Generated {DEMO['num_videos']} videos in {video_dir}")

### The VideoDataset class

One class handles the whole pipeline: metadata management and clip indexing at init, cached frame loading with a decode fallback, temporally consistent augmentation, and the PyTorch `Dataset` interface. Key design decisions:

- **Clip indexing** extracts up to 10 evenly spaced clips per video so long videos do not dominate sampling.
- **HDF5 caching** turns seconds of video decoding into milliseconds of loading after the first epoch.
- **Augmentation parameters are sampled once per clip** and applied to every frame, preserving motion and temporal coherence.
- **Corrupted videos return a random valid sample** instead of crashing training.


In [ ]:
import h5py
from torch.utils.data import Dataset, DataLoader


class VideoDataset(Dataset):
    """
    Efficient video dataset with intelligent caching and sampling.
    Handles videos of varying lengths and resolutions gracefully.
    """

    def __init__(self,
                 video_dir: str,
                 metadata_path: str,
                 cache_dir: str = None,
                 clip_length: int = 16,
                 frame_size: tuple = (256, 256),
                 frame_skip: int = 4,
                 augment: bool = True):

        self.video_dir = Path(video_dir)
        self.clip_length = clip_length
        self.frame_size = frame_size
        self.frame_skip = frame_skip  # Sample every Nth frame for temporal coverage
        self.augment = augment

        # Load metadata (video paths, captions, durations)
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)
        print(f"Loaded {len(self.metadata)} videos from metadata")

        # Initialize cache for preprocessed frames (100x speedup)
        self.cache_dir = Path(cache_dir) if cache_dir else None
        if self.cache_dir:
            self.cache_dir.mkdir(exist_ok=True, parents=True)

        # Build clip index for efficient sampling
        self.clip_index = self._build_clip_index()
        print(f"Built index with {len(self.clip_index)} valid clips")

    def _build_clip_index(self):
        """
        Build index of all valid clips across all videos.
        Handles varying video lengths by extracting multiple clips per video.
        """
        clip_index = []

        for video_info in self.metadata:
            video_path = self.video_dir / video_info['path']
            duration_frames = video_info['num_frames']

            # Calculate how many frames we need for one clip
            clip_frames = self.clip_length * self.frame_skip

            if duration_frames >= clip_frames:
                # Video is long enough for at least one clip
                max_start = duration_frames - clip_frames

                # Extract up to 10 clips per video for balanced sampling
                num_clips = min(10, max_start // clip_frames + 1)

                for i in range(num_clips):
                    # Distribute clips evenly across video duration
                    start_frame = int(i * max_start / max(num_clips - 1, 1))
                    clip_index.append({
                        'video_path': video_path,
                        'start_frame': start_frame,
                        'caption': video_info['caption'],
                        'video_id': video_info['id']
                    })

        return clip_index

    def _load_video_frames(self, video_path, start_frame):
        """
        Load frames with intelligent caching strategy.
        Falls back to video decoding if cache miss occurs.
        """
        # Check cache first (100x faster than decoding)
        cache_path = None
        if self.cache_dir:
            cache_key = f"{video_path.stem}_{start_frame}"
            cache_path = self.cache_dir / f"{cache_key}.h5"

            if cache_path.exists():
                with h5py.File(cache_path, 'r') as f:
                    frames = f['frames'][:]
                return torch.from_numpy(frames)

        # Fall back to video decoding
        cap = cv2.VideoCapture(str(video_path))
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frames = []
        for i in range(self.clip_length):
            # Skip frames according to frame_skip parameter
            for _ in range(self.frame_skip - 1):
                ret, _ = cap.read()
                if not ret:
                    cap.release()
                    return None  # Handle corrupted videos gracefully

            # Read the frame we actually keep
            ret, frame = cap.read()
            if not ret:
                cap.release()
                return None

            # Resize and convert color space
            frame = cv2.resize(frame, self.frame_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        # Normalize to [0, 1] range
        frames = np.stack(frames).astype(np.float32) / 255.0

        # Cache for future use
        if self.cache_dir:
            with h5py.File(cache_path, 'w') as f:
                f.create_dataset('frames', data=frames, compression='gzip')

        return torch.from_numpy(frames)

    def _augment_frames(self, frames):
        """
        Apply augmentations consistently across all frames.
        Maintains temporal coherence while increasing diversity.
        """
        if not self.augment:
            return frames

        B, H, W, C = frames.shape

        # Random horizontal flip (50% chance, same for all frames)
        if torch.rand(1) > 0.5:
            frames = frames.flip(dims=[2])

        # Random crop (90% of original size, same position for all frames)
        crop_size = int(H * 0.9)
        y = torch.randint(0, H - crop_size + 1, (1,)).item()
        x = torch.randint(0, W - crop_size + 1, (1,)).item()
        frames = frames[:, y:y+crop_size, x:x+crop_size]

        # Resize back to original dimensions
        frames = torch.nn.functional.interpolate(
            frames.permute(0, 3, 1, 2),  # BHWC -> BCHW for interpolation
            size=(H, W),
            mode='bilinear',
            align_corners=False
        ).permute(0, 2, 3, 1)  # Back to BHWC

        # Color jitter (plus or minus 10% brightness, consistent across frames)
        brightness = 1 + (torch.rand(1) - 0.5) * 0.2
        frames = frames * brightness
        frames = frames.clamp(0, 1)

        return frames

    def __len__(self):
        return len(self.clip_index)

    def __getitem__(self, idx):
        """Get a single training sample."""
        clip_info = self.clip_index[idx]

        # Load and decode frames
        frames = self._load_video_frames(
            clip_info['video_path'],
            clip_info['start_frame']
        )

        if frames is None:
            # Corrupted video: return random valid sample instead
            return self.__getitem__(torch.randint(len(self), (1,)).item())

        # Apply augmentations
        frames = self._augment_frames(frames)

        # Convert from THWC to TCHW format (channels first for GPU)
        frames = frames.permute(0, 3, 1, 2).unsqueeze(0)  # [1, T, C, H, W]

        return {
            'frames': frames,
            'caption': clip_info['caption'],
            'video_id': clip_info['video_id']
        }


# Efficient batching for DataLoader
def video_collate_fn(batch):
    """Combine samples into a batch."""
    frames = torch.cat([item['frames'] for item in batch], dim=0)
    captions = [item['caption'] for item in batch]
    video_ids = [item['video_id'] for item in batch]

    return {
        'frames': frames,  # [B, T, C, H, W]
        'captions': captions,  # List of strings
        'video_ids': video_ids  # List of IDs for tracking
    }

### Building the DataLoader

Wire the dataset to PyTorch's loader and inspect one batch. The frames tensor is 5D: `[Batch, Time, Channels, Height, Width]`.


In [ ]:
train_dataset = VideoDataset(
    video_dir=video_dir,
    metadata_path=metadata_path,
    cache_dir=WORK_DIR / "cache",
    clip_length=DEMO["clip_length"],
    frame_size=(DEMO["frame_size"], DEMO["frame_size"]),
    frame_skip=DEMO["frame_skip"],
    augment=True,
)

val_dataset = VideoDataset(
    video_dir=video_dir,
    metadata_path=metadata_path,
    cache_dir=WORK_DIR / "cache",
    clip_length=DEMO["clip_length"],
    frame_size=(DEMO["frame_size"], DEMO["frame_size"]),
    frame_skip=DEMO["frame_skip"],
    augment=False,  # No augmentation for validation
)

train_loader = DataLoader(
    train_dataset, batch_size=DEMO["batch_size"], shuffle=True,
    collate_fn=video_collate_fn, num_workers=0,
)
val_loader = DataLoader(
    val_dataset, batch_size=DEMO["batch_size"], shuffle=False,
    collate_fn=video_collate_fn, num_workers=0,
)

batch = next(iter(train_loader))
print(f"Frames batch shape [B, T, C, H, W]: {tuple(batch['frames'].shape)}")
print(f"Captions: {batch['captions']}")

### Multi-resolution training and temporal sampling

Two strategies from the chapter worth remembering, even though they need real data to demonstrate:

- **Resolution curriculum:** start training at low resolution (128 x 128) for fast, stable convergence, then raise it to the target (512 x 512 or higher). This accelerates training 3 to 5x versus fixed-resolution training. OpenSora and Hunyuan both use it.
- **Temporal sampling:** not all frames are equally informative. Uniform strides, random sampling, and motion-aware selection each trade off coverage against redundancy, and the choice separates smooth generated motion from jerky sequences.


## 3. Training Infrastructure and Optimization

Each training iteration follows the same cycle: load a batch, add noise at a random diffusion timestep, predict that noise, measure the error with MSE, backpropagate, clip gradients, and step the optimizer. The trainer below packages this loop with the memory and stability techniques video models need.

### TF32: Latte's performance secret

TF32 on NVIDIA Ampere or newer GPUs gives roughly a 2x matmul speedup while keeping FP32 storage. Latte uses this instead of full mixed precision for stability.


In [ ]:
# Enable TF32 as in Latte's train.py
# These two lines give ~2x speedup on A100/H100 GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Note: Latte disables these for FP16 training.
# Set to False if using mixed precision.

### The VideoModelTrainer

The full trainer, assembled from the chapter's seven parts:

1. **Initialization:** device setup, optimizer, scheduler, optional AMP scaler, optional EMA model.
2. **Distributed setup:** DDP over NCCL when running multi-GPU.
3. **Optimizer and scheduler:** AdamW with no weight decay on biases and norms, linear warmup then cosine annealing.
4. **Gradient accumulation:** micro-batches simulate large batches, with `no_sync()` deferring gradient communication.
5. **Training step:** forward, loss, backward, clip, step, EMA update.
6. **Loss computation:** sample a timestep, add noise, predict it, MSE. The noise schedule here is the chapter's simplified linear one; production code uses a proper cosine schedule.
7. **Epoch loop:** logging, validation, and full-state checkpointing so training can resume exactly where it left off.


In [ ]:
import copy
from contextlib import contextmanager

import torch.nn as nn
import torch.distributed as dist
from torch.cuda.amp import autocast, GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP


class VideoModelTrainer:
    """
    Educational trainer for video generation models.
    Handles distributed training, memory optimization, and checkpointing.
    """

    def __init__(self,
                 model: nn.Module,
                 train_dataloader,
                 val_dataloader=None,
                 learning_rate: float = 1e-4,
                 warmup_steps: int = 10000,
                 total_steps: int = 1000000,
                 num_timesteps: int = 1000,
                 gradient_clip: float = 1.0,
                 ema_decay: float = 0.9999,
                 mixed_precision: bool = True,
                 gradient_accumulation_steps: int = 1,
                 checkpoint_dir: str = "./checkpoints",
                 distributed: bool = False):

        # Core components
        self.model = model
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader

        # Training configuration
        self.gradient_clip = gradient_clip
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.mixed_precision = mixed_precision
        self.distributed = distributed
        self.num_timesteps = num_timesteps
        self.ema_decay = ema_decay

        # Setup device and distributed training
        if distributed:
            self.setup_distributed()
        else:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.model = self.model.to(self.device)

        # Initialize training components
        self.optimizer = self._build_optimizer(learning_rate)
        self.scheduler = self._build_scheduler(warmup_steps, total_steps)
        self.scaler = GradScaler() if mixed_precision else None
        self.ema_model = self._build_ema_model(ema_decay) if ema_decay else None

        # Training state tracking
        self.global_step = 0
        self.epoch = 0
        self.best_val_loss = float('inf')

        # Checkpointing setup
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True, parents=True)

    # Part 2: Distributed training configuration
    def setup_distributed(self):
        """Initialize distributed training across multiple GPUs."""
        # Initialize process group for inter-GPU communication
        dist.init_process_group(backend='nccl')  # NCCL optimized for NVIDIA GPUs

        self.rank = dist.get_rank()  # This GPU's ID
        self.world_size = dist.get_world_size()  # Total number of GPUs
        self.device = torch.device(f'cuda:{self.rank}')

        # Ensure each process uses its assigned GPU
        torch.cuda.set_device(self.device)
        self.model = self.model.to(self.device)

        # Wrap model for distributed training
        self.model = DDP(self.model, device_ids=[self.rank])

    # Part 3: Optimizer and scheduler configuration
    def _build_optimizer(self, learning_rate):
        """Build AdamW with selective weight decay."""
        # Separate parameters by weight decay requirement
        decay_params = []
        no_decay_params = []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            # Biases and norm parameters should not be regularized
            if 'bias' in name or 'norm' in name:
                no_decay_params.append(param)
            else:
                decay_params.append(param)

        # AdamW with different weight decay for different params
        optimizer = torch.optim.AdamW([
            {'params': decay_params, 'weight_decay': 0.01},
            {'params': no_decay_params, 'weight_decay': 0.0}
        ], lr=learning_rate, betas=(0.9, 0.999), eps=1e-8)

        print(f"Optimizer: {len(decay_params)} params with decay, "
              f"{len(no_decay_params)} params without decay")

        return optimizer

    def _build_scheduler(self, warmup_steps, total_steps=1000000):
        """Cosine annealing with linear warmup."""
        warmup_steps = max(warmup_steps, 1)

        def lr_lambda(step):
            if step < warmup_steps:
                # Linear warmup
                return step / warmup_steps
            # Cosine annealing after warmup
            progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
            return 0.5 * (1 + np.cos(np.pi * progress))

        return torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)

    def _build_ema_model(self, decay):
        """Initialize EMA model for stable generation."""
        ema_model = copy.deepcopy(self.model)
        ema_model.eval()
        ema_model.requires_grad_(False)
        return ema_model

    # Part 4: Gradient accumulation and memory management
    @contextmanager
    def _gradient_accumulation_context(self, step_idx):
        """
        Control gradient synchronization in distributed training.
        Key insight: only communicate gradients when ready to update.
        """
        if self.distributed:
            if step_idx < self.gradient_accumulation_steps - 1:
                # Accumulate locally - no inter-GPU communication
                with self.model.no_sync():
                    yield
            else:
                # Final accumulation - sync gradients across GPUs
                yield
        else:
            yield

    def _split_batch(self, batch, num_splits):
        """Split batch into micro-batches for accumulation."""
        batch_size = len(batch['frames'])
        micro_size = max(batch_size // num_splits, 1)

        micro_batches = []
        for i in range(num_splits):
            start = i * micro_size
            if start >= batch_size:
                break
            end = start + micro_size if i < num_splits - 1 else batch_size
            micro_batches.append({
                'frames': batch['frames'][start:end],
                'captions': batch['captions'][start:end]
            })
        return micro_batches

    # Part 5: Core training step implementation
    def train_step(self, batch):
        """Execute one complete training step."""
        total_loss = 0

        # Split batch for gradient accumulation
        micro_batches = self._split_batch(batch, self.gradient_accumulation_steps)

        # Process each micro-batch
        for i, micro_batch in enumerate(micro_batches):
            with self._gradient_accumulation_context(i):
                if self.mixed_precision:
                    # FP16 computation with automatic scaling
                    with autocast():
                        loss = self._compute_loss(micro_batch)

                    # Scale loss by accumulation steps
                    loss = loss / self.gradient_accumulation_steps

                    # Backward with gradient scaling for FP16
                    self.scaler.scale(loss).backward()
                else:
                    # Standard FP32 computation
                    loss = self._compute_loss(micro_batch)
                    loss = loss / self.gradient_accumulation_steps
                    loss.backward()

                total_loss += loss.item()

        # Gradient clipping prevents exploding gradients
        if self.mixed_precision:
            # Unscale gradients before clipping
            self.scaler.unscale_(self.optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(),
            self.gradient_clip
        )

        # Optimizer step with mixed precision handling
        if self.mixed_precision:
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            self.optimizer.step()

        # Clear gradients (set_to_none saves memory)
        self.optimizer.zero_grad(set_to_none=True)
        self.scheduler.step()

        # Update EMA model if configured
        if self.ema_model:
            self._update_ema()

        self.global_step += 1

        return {
            'loss': total_loss,
            'grad_norm': grad_norm.item(),
            'learning_rate': self.scheduler.get_last_lr()[0]
        }

    # Part 6: Loss computation and model updates
    def _compute_loss(self, batch):
        """
        Compute diffusion loss for video generation.
        Trains the model to predict noise added at various timesteps.
        """
        frames = batch['frames'].to(self.device)
        captions = batch['captions']

        # Sample random timesteps for each video
        batch_size = frames.shape[0]
        timesteps = torch.randint(
            0, self.num_timesteps,  # Diffusion timesteps
            (batch_size,),
            device=self.device
        )

        # Generate noise and create noisy videos
        noise = torch.randn_like(frames)

        # Forward diffusion: add noise according to schedule
        noisy_frames = self.add_noise(frames, noise, timesteps)

        # Model predicts the noise that was added
        predicted_noise = self.model(
            noisy_frames,
            timesteps,
            captions
        )

        # MSE loss between predicted and actual noise
        loss = torch.nn.functional.mse_loss(predicted_noise, noise)

        return loss

    def add_noise(self, x, noise, t):
        """Add noise to frames using diffusion schedule."""
        # Simplified linear schedule (replace with cosine for production)
        alpha = (1 - t.float() / self.num_timesteps).view(-1, 1, 1, 1, 1)
        return alpha * x + (1 - alpha).sqrt() * noise

    def _update_ema(self):
        """
        Update exponential moving average model.
        EMA provides stable weights for generation.
        """
        with torch.no_grad():
            decay = self.ema_decay
            for ema_param, model_param in zip(
                self.ema_model.parameters(),
                self.model.parameters()
            ):
                # Weighted average: mostly old weights, small contribution from new
                ema_param.data = decay * ema_param.data + (1 - decay) * model_param.data

    # Part 7: Training loop and monitoring
    def train_epoch(self):
        """Complete one training epoch with monitoring."""
        self.model.train()
        epoch_metrics = []

        for batch_idx, batch in enumerate(self.train_dataloader):
            # Execute training step
            metrics = self.train_step(batch)
            epoch_metrics.append(metrics)

            # Periodic logging (every 100 steps)
            if self.global_step % 100 == 0:
                print(f"Step {self.global_step}: Loss={metrics['loss']:.4f}, "
                      f"Grad={metrics['grad_norm']:.4f}, LR={metrics['learning_rate']:.2e}")

            # Periodic validation (every 1000 steps)
            if self.global_step % 1000 == 0 and self.val_dataloader:
                val_metrics = self.validate()
                if val_metrics['loss'] < self.best_val_loss:
                    self.best_val_loss = val_metrics['loss']
                    self.save_checkpoint('best')

            # Regular checkpointing (every 5000 steps)
            if self.global_step % 5000 == 0:
                self.save_checkpoint(f'step_{self.global_step}')

        self.epoch += 1
        return epoch_metrics

    def validate(self):
        """Run validation and return metrics."""
        self.model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in self.val_dataloader:
                loss = self._compute_loss(batch)
                val_losses.append(loss.item())

        self.model.train()
        return {'loss': np.mean(val_losses)}

    def save_checkpoint(self, name):
        """Save training state for resume."""
        checkpoint = {
            'model': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict(),
            'ema_model': self.ema_model.state_dict() if self.ema_model else None,
            'scaler': self.scaler.state_dict() if self.scaler else None,
            'global_step': self.global_step,
            'epoch': self.epoch,
            'best_val_loss': self.best_val_loss
        }
        torch.save(checkpoint, self.checkpoint_dir / f'{name}.pt')

### Running the trainer on a toy model

The real Latte model from Chapter 3 is far too large for a quick demo, so we stand in a tiny 3D convolutional denoiser with the same interface: it takes noisy frames, timesteps, and captions, and predicts the noise. This lets the full trainer run end to end in about a minute on CPU.


In [ ]:
class TinyVideoDenoiser(nn.Module):
    """Minimal stand-in for the Latte model: predicts noise from noisy frames."""

    def __init__(self, channels=3, hidden=16):
        super().__init__()
        self.hidden = hidden
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, hidden)
        )
        self.conv_in = nn.Conv3d(channels, hidden, 3, padding=1)
        self.conv_mid = nn.Conv3d(hidden, hidden, 3, padding=1)
        self.conv_out = nn.Conv3d(hidden, channels, 3, padding=1)

    def forward(self, x, timesteps, captions=None):
        # x: [B, T, C, H, W] -> [B, C, T, H, W] for Conv3d
        h = x.permute(0, 2, 1, 3, 4)
        temb = self.time_embed(timesteps.float().view(-1, 1) / 1000.0)
        h = torch.nn.functional.silu(
            self.conv_in(h) + temb.view(-1, self.hidden, 1, 1, 1)
        )
        h = torch.nn.functional.silu(self.conv_mid(h))
        out = self.conv_out(h)
        return out.permute(0, 2, 1, 3, 4)  # Back to [B, T, C, H, W]


model = TinyVideoDenoiser()
num_params = sum(p.numel() for p in model.parameters())
print(f"Toy model parameters: {num_params:,}")

trainer = VideoModelTrainer(
    model=model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    learning_rate=DEMO["learning_rate"],
    warmup_steps=DEMO["warmup_steps"],
    total_steps=DEMO["train_steps"],
    gradient_clip=1.0,
    ema_decay=0.999,
    mixed_precision=False,  # TF32 path, following Latte; flip on for FP16
    gradient_accumulation_steps=DEMO["grad_accum"],
    checkpoint_dir=WORK_DIR / "checkpoints",
)

In [ ]:
# Run a short training demo: DEMO["train_steps"] optimizer steps.
# With real data and the real model this loop runs for ~1M steps.
trainer.model.train()
steps_done = 0
while steps_done < DEMO["train_steps"]:
    for batch in train_loader:
        metrics = trainer.train_step(batch)
        steps_done += 1
        print(f"step {trainer.global_step:3d} | loss {metrics['loss']:.4f} "
              f"| grad_norm {metrics['grad_norm']:.3f} "
              f"| lr {metrics['learning_rate']:.2e}")
        if steps_done >= DEMO["train_steps"]:
            break

val_metrics = trainer.validate()
print(f"\nValidation loss: {val_metrics['loss']:.4f}")

trainer.save_checkpoint("demo_final")
print(f"Checkpoint saved to {trainer.checkpoint_dir / 'demo_final.pt'}")

### Learning rate scheduling and warmup

Video models are unstable early in training, so the schedule warms up linearly from zero, then decays with cosine annealing. Here is the exact schedule the trainer built, plotted over a full-length run.


In [ ]:
import matplotlib.pyplot as plt

warmup, total = 10_000, 1_000_000
steps = np.arange(0, total, 1000)
lrs = np.where(
    steps < warmup,
    steps / warmup,
    0.5 * (1 + np.cos(np.pi * (steps - warmup) / (total - warmup))),
) * 1e-4

plt.figure(figsize=(8, 3))
plt.plot(steps, lrs)
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.title("Linear warmup + cosine annealing (Latte-style schedule)")
plt.tight_layout()
plt.show()

### Gradient management and stability

The temporal dimension creates long-range dependencies that make gradients explode or vanish. The trainer already applies the main defense, clipping the gradient norm to 1.0. Skip connections, careful depth-scaled initialization, and gradient penalties are additional tools when training misbehaves.


## 4. Loss Functions and Metrics

The chapter's analysis of production systems from 2022 to 2025 (CogVideo, Latte, Open-Sora, HunyuanVideo, Wan2.2) finds they all rely on MSE variants: plain MSE, SNR-weighted MSE, or flow matching, which is still MSE on a velocity target. Start with Latte's plain MSE and add complexity only with evidence it helps.

### Simple MSE loss (what Latte actually uses)


In [ ]:
import torch.nn.functional as F


def training_step(model, diffusion, x_latent, device, model_kwargs=None):
    """
    Latte's actual training step from train.py.
    The entire loss is just MSE between predicted and target noise.
    """
    model_kwargs = model_kwargs or {}

    # Sample random timesteps for each video in the batch
    t = torch.randint(0, diffusion.num_timesteps, (x_latent.shape[0],), device=device)

    # Get loss from diffusion.training_losses (which internally uses MSE)
    loss_dict = diffusion.training_losses(model, x_latent, t, model_kwargs)

    # Extract the loss - it is just MSE, nothing more
    loss = loss_dict["loss"].mean()

    return loss


# The core of that objective, demonstrated directly:
pred_noise = torch.randn(2, 8, 3, 16, 16)
true_noise = torch.randn(2, 8, 3, 16, 16)
print(f"Plain MSE loss: {F.mse_loss(pred_noise, true_noise).item():.4f}")

### Optional: Min-SNR weighting

If the model struggles with fine details, Min-SNR weighting rebalances learning across noise levels. CogVideo pioneered this in 2022; Latte later showed you can often do without it.


In [ ]:
# Min-SNR weighting - only add if you observe learning imbalance
def compute_loss_with_min_snr(pred_noise, target_noise, timesteps, alphas_cumprod, gamma=5.0):
    """
    MSE loss with Min-SNR weighting to balance learning.
    Only use if simple MSE shows poor detail generation.
    """
    # Start with base MSE
    mse_loss = F.mse_loss(pred_noise, target_noise, reduction='none')

    # Calculate signal-to-noise ratio
    snr = alphas_cumprod[timesteps] / (1 - alphas_cumprod[timesteps] + 1e-8)

    # Apply Min-SNR weighting
    min_snr = torch.minimum(snr, torch.ones_like(snr) * gamma)
    weight = min_snr / snr

    # Weight the loss
    weighted_loss = mse_loss * weight.view(-1, 1, 1, 1, 1)

    return weighted_loss.mean()


# Build alphas_cumprod from the config's linear beta schedule and try it out
betas = torch.linspace(0.0001, 0.02, 1000)
alphas_cumprod = torch.cumprod(1.0 - betas, dim=0)

timesteps = torch.randint(0, 1000, (2,))
loss = compute_loss_with_min_snr(pred_noise, true_noise, timesteps, alphas_cumprod)
print(f"Min-SNR weighted loss at t={timesteps.tolist()}: {loss.item():.4f}")

# Visualize how the weight suppresses easy low-noise timesteps
snr = alphas_cumprod / (1 - alphas_cumprod + 1e-8)
weight = torch.minimum(snr, torch.full_like(snr, 5.0)) / snr
plt.figure(figsize=(8, 3))
plt.plot(weight.numpy())
plt.xlabel("Diffusion timestep t")
plt.ylabel("Loss weight")
plt.title("Min-SNR weighting (gamma=5) across timesteps")
plt.tight_layout()
plt.show()

## 5. Scaling and Distributed Training

Four parallelism strategies power modern training: data parallelism (each GPU holds the full model, different data), model parallelism (layers split across GPUs), pipeline parallelism (micro-batches flow through model segments), and 3D parallelism combining all three. Latte gets state-of-the-art results with plain data parallelism because its architecture fits on a single GPU.

### Data parallelism, the Latte approach

The setup below runs under `torchrun`, which sets the `RANK` and `LOCAL_RANK` environment variables. We define it here but only execute it in a real multi-GPU launch.


In [ ]:
# Latte's distributed setup from train.py
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def setup_distributed():
    """Minimal distributed setup following Latte."""
    dist.init_process_group("nccl")
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device("cuda", local_rank)
    torch.cuda.set_device(device)
    return rank, local_rank, device


# Usage in a training script launched with:
#   torchrun --nproc_per_node=8 train.py
#
# rank, local_rank, device = setup_distributed()
# model = model.to(device)
# model = DDP(model, device_ids=[local_rank])

print("setup_distributed defined; launch with torchrun to use it")

### When to reach for more than DDP

Consider model, pipeline, or 3D parallelism only when: the model exceeds 24 GB VRAM even with gradient checkpointing, a batch size of 1 still runs out of memory, DDP scaling has plateaued, or you have the engineering budget to debug distributed systems. Keep communication simple (NCCL AllReduce) and checkpoint everything, every hour at minimum.


## Summary

You built the complete training pipeline for a video generation model:

- A YAML plus OmegaConf configuration system that makes every experiment reproducible.
- A video dataset with clip indexing, HDF5 caching, and temporally consistent augmentation.
- A trainer with warmup plus cosine scheduling, gradient accumulation, clipping, EMA weights, and full-state checkpointing.
- Latte's plain MSE diffusion loss, with Min-SNR weighting held in reserve.
- Data-parallel scaling with DDP, and the criteria for when heavier parallelism is worth it.

The key insight: production models succeed through engineering excellence, not algorithmic complexity. Clean data, simple losses, and stable training beat exotic tricks. Chapter 5 builds on this foundation with fine-tuning strategies that adapt pretrained models to specific creative tasks.
